In [0]:
# ==========================================
# 1️⃣ Create Widgets & Validate
# ==========================================

dbutils.widgets.text("table_metadata", "")
dbutils.widgets.text("catalog_name", "")

table_metadata_str = dbutils.widgets.get("table_metadata")
catalog_name = dbutils.widgets.get("catalog_name").strip()

print(f"Table Metadata Passed: {table_metadata_str}")

if not table_metadata_str:
    raise ValueError("table_metadata widget cannot be empty")

if not catalog_name:
    raise ValueError("catalog_name widget cannot be empty (e.g. 'banking' — see config/*.json)")

import ast
table_metadata = ast.literal_eval(table_metadata_str)

table_id = table_metadata.get("table_id")
if not table_id:
    raise ValueError("table_id not found in table_metadata")

table_id = int(table_id)


# ==========================================
# 2️⃣ Read Table Parameters
# ==========================================

params_df = (
    spark.table(f"{catalog_name}.metadata.table_parameters")
         .filter(f"table_id = {table_id}")
)

# ==========================================
# 3️⃣ Convert to Single JSON Object
# ==========================================

rows = params_df.select("parameter_name", "parameter_value").collect()

parameters_dict = {
    row.parameter_name: row.parameter_value
    for row in rows
}

print("Parameters JSON Object:")
print(parameters_dict)


# ==========================================
# 4️⃣ Set Databricks Task Value
# ==========================================

dbutils.jobs.taskValues.set(
    key="table_parameters",
    value=parameters_dict
)

print("Task value 'table_parameters' has been set.")